# Assignment 2: Bigram Language Model and Generative Pretrained Transformer (GPT)


The objective of this assignment is to train a simplified transformer model. The primary differences between the implementation:
* tokenizer (we use a character level encoder simplicity and compute constraints)
* size (we are using 1 consumer grade gpu hosted on colab and a small dataset. in practice, the models are much larger and are trained on much more data)
* efficiency


Most modern LLMs have multiple training stages, so we won't get a model that is capable of replying to you yet. However, this is the first step towards a model like ChatGPT and Llama.




In [ ]:
%env CUDA_LAUNCH_BLOCKING=1

import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
from dataclasses import dataclass


env: CUDA_LAUNCH_BLOCKING=1


## Part 1: Bigram MLP for TinyShakespeare (35 points)

1a) (1 point). Create a list `chars` that contains all unique characters in `text`

1b) (2 points). Implement `encode(s: str) -> list[int]`

1c) (2 points). Implement `decode(ids: list[int]) -> str`

1d) (5 points). Create two tensors, `inputs_one_hot` and `outputs_one_hot`. Use one hot encoding. Make sure to get every consecutive pair of characters. For example, for the word 'hello', we should create the following input-output pairs
```
he
el
ll
lo
```

1e) (10 points). Implement BigramOneHotMLP, a 2 layer MLP that predicts the next token. Specifically, implement the constructor, forward, and generate. The output dimension of the first layer should be 8. Use `torch.optim`. The activation function for the first layer should be `nn.LeakyReLU()`

Note: Use the `torch.nn.function.cross_entropy` loss. Read the [docs](https://pytorch.org/docs/stable/generated/torch.nn.functional.cross_entropy.html) about how this loss function works. The logits are the output of a network WITHOUT an activation function applied to the last layer. There are activation functions are applied to every layer except the last.

1f) (5 points). Train the BigramOneHotMLP for 1000 steps.

1g) (5 points). Create two tensors, `input_ids` and `outputs_one_hot`. These `input_ids` will be used for the embedding layer.

1h) (5 points). Implement and train BigramEmbeddingMLP, a 2 layer mlp that predicts the next token. Specifically, implement the constructor, forward, and generate functions. The output dimension of the first layer should be 8. Use `torch.optim`.



Note: the output will look like gibberish


In [ ]:
!wget https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt

--2025-06-21 15:56:00--  https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 1115394 (1.1M) [text/plain]
Saving to: ‘input.txt’

input.txt           100%[===================>]   1.06M  --.-KB/s    in 0.006s  

2025-06-21 15:56:00 (166 MB/s) - ‘input.txt’ saved [1115394/1115394]



In [ ]:
# For the bigram model, let's use the first 1000 characters for the data

with open('input.txt', 'r') as f:
  text = f.read()
text = text[:1000]

In [ ]:
#1a - Create a list chars that contains all unique characters in text
chars = sorted(list(set(text)))
vocab_size = len(chars)
print(chars)

['\n', ' ', '!', "'", ',', '.', ':', ';', '?', 'A', 'B', 'C', 'F', 'I', 'L', 'M', 'N', 'O', 'R', 'S', 'W', 'Y', 'a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'j', 'k', 'l', 'm', 'n', 'o', 'p', 'r', 's', 't', 'u', 'v', 'w', 'y', 'z']


In [ ]:
#1b) Implement encode(s: str) -> list[int]
def encode(s: str) -> list[int]:
    return [chars.index(c) for c in s]

#1c) Implement decode(ids: list[int]) -> str
def decode(ids: list[int]) -> str:
    return ''.join([chars[i] for i in ids])

#1d) Create two tensors, inputs_one_hot and outputs_one_hot. Use one hot encoding. Make sure to get every consecutive pair of characters. For example, for the word 'hello', we should create the following input-output pairs
def create_one_hot_inputs_and_outputs() -> list[torch.tensor, torch.tensor]:
    input_x, output_y = [], []
    for i in range(len(text) - 1):
        input_x.append(chars.index(text[i]))
        output_y.append(chars.index(text[i + 1]))

    x = torch.tensor(input_x, dtype=torch.long)
    y = torch.tensor(output_y, dtype=torch.long)

    inputs_one_hot = torch.nn.functional.one_hot(x, num_classes=len(chars)).float()
    outputs_one_hot = torch.nn.functional.one_hot(y, num_classes=len(chars)).float()

    return inputs_one_hot, outputs_one_hot

inputs_one_hot, outputs_one_hot = create_one_hot_inputs_and_outputs()

#1e) Implement BigramOneHotMLP, a 2 layer MLP that predicts the next token. Specifically, implement the constructor, forward, and generate. The output dimension of the first layer should be 8. Use torch.optim. The activation function for the first layer should be nn.LeakyReLU()
class BigramOneHotMLP(nn.Module):
    def __init__(self):
        super().__init__()
        # Two-layer neural network with 8 hidden units and LeakyReLU activation
        self.net = nn.Sequential(
            nn.Linear(len(chars), 8),
            nn.LeakyReLU(),
            nn.Linear(8, len(chars)))

    def forward(self, x):
        return self.net(x)

    def generate(self, start='a', max_new_tokens=100) -> str:
        # Generates text starting from the given character
        generated = [chars.index(start)]
        for _ in range(max_new_tokens):
            x = torch.nn.functional.one_hot(
                torch.tensor([generated[-1]]), num_classes=len(chars)).float()
            logits = self.forward(x)
            probs = torch.softmax(logits, dim=-1)
            next_id = torch.multinomial(probs, num_samples=1).item()
            generated.append(next_id)
        return decode(generated)

bigram_one_hot_mlp = BigramOneHotMLP()

#1f) Train the BigramOneHotMLP for 1000 steps.
optimizer = torch.optim.Adam(bigram_one_hot_mlp.parameters(), lr=1e-2)
loss_fn = torch.nn.functional.cross_entropy

# Training loop
for _ in range(1000):
    logits = bigram_one_hot_mlp(inputs_one_hot)
    loss = loss_fn(logits, outputs_one_hot.argmax(dim=1))
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    if _ % 100 == 0:
        print(f"Step {_} / 1000, Loss: {loss.item():.4f}")

# Generate text
print(bigram_one_hot_mlp.generate())


Step 0 / 1000, Loss: 3.9497
Step 100 / 1000, Loss: 2.3968
Step 200 / 1000, Loss: 2.2101
Step 300 / 1000, Loss: 2.1619
Step 400 / 1000, Loss: 2.1399
Step 500 / 1000, Loss: 2.1281
Step 600 / 1000, Loss: 2.1192
Step 700 / 1000, Loss: 2.1121
Step 800 / 1000, Loss: 2.1066
Step 900 / 1000, Loss: 2.1023
akinengomisthe d Cit s gh
Fist out k Lened gonore.
Wemy t, it weatorists our
Yo or anonizengous hizer


In [ ]:
#1g) Create two tensors, input_ids and outputs_one_hot. These input_ids will be used for the embedding layer.
def create_embedding_inputs_and_outputs() -> list[torch.tensor, torch.tensor]:
    input_x, output_y = [], []
    for i in range(len(text) - 1):
        input_x.append(chars.index(text[i]))
        output_y.append(chars.index(text[i + 1]))

    input_ids = torch.tensor(input_x, dtype=torch.long)
    outputs_one_hot = torch.nn.functional.one_hot(
        torch.tensor(output_y, dtype=torch.long),
        num_classes=len(chars)
    ).float()

    return input_ids, outputs_one_hot

input_ids, outputs_one_hot = create_embedding_inputs_and_outputs()
#1h) Implement and train BigramEmbeddingMLP, a 2 layer mlp that predicts the next token. Specifically, implement the constructor, forward, and generate functions. The output dimension of the first layer should be 8. Use torch.optim.
class BigramEmbeddingMLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.embedding = nn.Embedding(len(chars), len(chars))
        self.net = nn.Sequential(
            nn.Linear(len(chars), 8),
            nn.LeakyReLU(),
            nn.Linear(8, len(chars)))

    def forward(self, x):
        x_embed = self.embedding(x)
        return self.net(x_embed)

    def generate(self, start='a', max_new_tokens=100) -> str:
        generated = [chars.index(start)]
        for _ in range(max_new_tokens):
            x = torch.tensor([generated[-1]], dtype=torch.long)
            logits = self.forward(x)
            probs = torch.softmax(logits, dim=-1)
            next_id = torch.multinomial(probs, num_samples=1).item()
            generated.append(next_id)
        return decode(generated)

bigram_embedding_mlp = BigramEmbeddingMLP()
optimizer = torch.optim.Adam(bigram_embedding_mlp.parameters(), lr=1e-2)
loss_fn = torch.nn.functional.cross_entropy

# training loop
for _ in range(1000):
  logits = bigram_embedding_mlp(input_ids)
  loss = loss_fn(logits, outputs_one_hot.argmax(dim=1))
  optimizer.zero_grad()
  loss.backward()
  optimizer.step()
  if _ % 100 == 0:
    print(f"Step {_} / 1000, Loss: {loss.item():.4f}")
print(bigram_embedding_mlp.generate())

Step 0 / 1000, Loss: 3.8933
Step 100 / 1000, Loss: 2.2680
Step 200 / 1000, Loss: 2.1778
Step 300 / 1000, Loss: 2.1410
Step 400 / 1000, Loss: 2.1244
Step 500 / 1000, Loss: 2.1177
Step 600 / 1000, Loss: 2.1123
Step 700 / 1000, Loss: 2.1092
Step 800 / 1000, Loss: 2.1041
Step 900 / 1000, Loss: 2.1018
anneans ansol: aw'tht fod, an ghisoo ceiress, at is't kne ore.
Fize s t bren
Ald mawey amet thigurs g


## Part 2: Generative Pretrained Transformer (65 points)

For this part, it is best to use a gpu. In the settings at the top go to Runtime -> Change Runtime Type and select T4 GPU

In [ ]:
# run nvidia-smi to check gpu usage
!nvidia-smi

Sat Jun 21 15:56:29 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.54.15              Driver Version: 550.54.15      CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   44C    P8              9W /   70W |       2MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
# !wget https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt

In [ ]:
# For the gpt model, using the full text
with open('input.txt', 'r') as f:
    text = f.read()

Implement a character level tokenization function.

1. Create a list of unique characters in the string. (1 points)
2. Implement a function `encode(s: str) -> list[int]` that takes a string and returns a list of ids (1 point)
3. Implement a function `decode(ids: list[int]) -> str` that takes a list of ids (ints) and returns a string (1 point)


In [ ]:
#1. Create a list of unique characters in the string.
chars = sorted(list(set(text)))
vocab_size = len(chars)

#2. Implement a function encode(s: str) -> list[int] that takes a string and returns a list of ids
def encode(s: str) -> list[int]:
    return [chars.index(c) for c in s]

#3. Implement a function decode(ids: list[int]) -> str that takes a list of ids (ints) and returns a string
def decode(ids: list[int]) -> str:
    return ''.join([chars[i] for i in ids])

In [ ]:
data = torch.tensor(encode(text), dtype=torch.long).cuda()

In [ ]:
block_size = 16
data[:block_size+1]

tensor([18, 47, 56, 57, 58,  1, 15, 47, 58, 47, 64, 43, 52, 10,  0, 14, 43],
       device='cuda:0')

To train a transformer, we feed the model `n` tokens (context) and try to predict the `n+1`th token (target) in the sequence.



In [ ]:
x = data[:block_size]
y = data[1:block_size+1]
for t in range(block_size):
    context = x[:t+1]
    target = y[t]
    print(f"when input is {context} the target: {target}")

when input is tensor([18], device='cuda:0') the target: 47
when input is tensor([18, 47], device='cuda:0') the target: 56
when input is tensor([18, 47, 56], device='cuda:0') the target: 57
when input is tensor([18, 47, 56, 57], device='cuda:0') the target: 58
when input is tensor([18, 47, 56, 57, 58], device='cuda:0') the target: 1
when input is tensor([18, 47, 56, 57, 58,  1], device='cuda:0') the target: 15
when input is tensor([18, 47, 56, 57, 58,  1, 15], device='cuda:0') the target: 47
when input is tensor([18, 47, 56, 57, 58,  1, 15, 47], device='cuda:0') the target: 58
when input is tensor([18, 47, 56, 57, 58,  1, 15, 47, 58], device='cuda:0') the target: 47
when input is tensor([18, 47, 56, 57, 58,  1, 15, 47, 58, 47], device='cuda:0') the target: 64
when input is tensor([18, 47, 56, 57, 58,  1, 15, 47, 58, 47, 64], device='cuda:0') the target: 43
when input is tensor([18, 47, 56, 57, 58,  1, 15, 47, 58, 47, 64, 43], device='cuda:0') the target: 52
when input is tensor([18, 47,

In [ ]:
batch_size = 64
device = 'cuda' if torch.cuda.is_available() else 'cpu'
def get_batch():
    ix = torch.randint(len(data) - block_size, (batch_size,))
    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    x, y = x.to(device), y.to(device)
    return x, y

### Single Self Attention Head (5 points)
![](https://i.ibb.co/GWR1XG0/head.png)

In [ ]:
block_size = 16
n_embd = 32
n_head = 4
head_size = n_embd // n_head

In [ ]:
# Single Self Attention Head
class SelfAttentionHead(nn.Module):
    def __init__(self, head_size, n_embd):
        super().__init__()
        self.key = nn.Linear(n_embd, head_size, bias=False)
        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.value = nn.Linear(n_embd, head_size, bias=False)

        # Lower triangular matrix to mask future tokens
        self.register_buffer("tril", torch.tril(torch.ones(block_size, block_size)))
        self.dropout = nn.Dropout(0.1)

    def forward(self, x):
        B, T, C = x.shape
        k = self.key(x)
        q = self.query(x)

        # calculating attention scores
        wei = q @ k.transpose(-2, -1) / (k.shape[-1] ** 0.5)  # (B, T, T)
        wei = wei.masked_fill(self.tril[:T, :T] == 0, float('-inf'))
        wei = F.softmax(wei, dim=-1)
        wei = self.dropout(wei)

        # calculating attention output
        v = self.value(x)
        out = wei @ v
        return out

### Multihead Self Attention (5 points)

`constructor`

- Create 4 `SelfAttentionHead` instances. Consider using `nn.ModuleList`
- Create a linear layer with n_embd input dim and n_embd output dim

`forward`

In the forward implementation, pass `x` through each head, then concatenate all the outputs along the feature dimension, then pass the concatenated output through the linear layer

![](https://i.ibb.co/y5SwyZZ/multihead.png)

In [ ]:
# Multihead Self Attention
class MultiHeadAttention(nn.Module):
    def __init__(self, num_heads, head_size, n_embd):
        super().__init__()
        self.heads = nn.ModuleList([SelfAttentionHead(head_size, n_embd) for _ in range(num_heads)])
        self.proj = nn.Linear(num_heads * head_size, n_embd)
        self.dropout = nn.Dropout(0.1)

    #apply heads and add the results
    def forward(self, x):
        out = torch.cat([head(x) for head in self.heads], dim=-1) # (B, T, num_heads * head_size)
        out = self.proj(out)
        out = self.dropout(out)
        return out

## MLP (2 points)
Implement a 2 layer MLP


![](https://i.ibb.co/C0DtrF5/ff.png)

In [ ]:
#MLP
class MLP(nn.Module):
    def __init__(self, n_embd):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embd, 4 * n_embd),  # Expand embedding size
            nn.ReLU(),                     # Non-linearity
            nn.Linear(4 * n_embd, n_embd), # Project back to original size
            nn.Dropout(0.1)               # Regularization
        )

    def forward(self, x):
        return self.net(x)

## Transformer block (20 points)

Layer normalization help training stability by normalizing the outputs of neurons within a single layer across all features for each individual data point, not across a full batch or a specific feature.

Dropout is a form of regularization to prevent overfitting.

This is the diagram of a transformer block:

![](https://i.ibb.co/X85C473/block.png)

In [ ]:
#Transformer Block
class Block(nn.Module):
    def __init__(self, n_embd, n_head):
        super().__init__()
        head_size = n_embd // n_head
        self.sa = MultiHeadAttention(n_head, head_size, n_embd) # Multi-head self-attention
        self.ffwd = MLP(n_embd) # Feedforward network
        self.ln1 = nn.LayerNorm(n_embd) # LayerNorm before attention
        self.ln2 = nn.LayerNorm(n_embd) # LayerNorm before MLP

    def forward(self, x):
        x = x + self.sa(self.ln1(x))
        x = x + self.ffwd(self.ln2(x))
        return x

## GPT

`constructor` (5 points)

1. create the token embedding table and the position embedding table
2. create variable `self.blocks` that is a series of 4 `Block`s. The data will pass through each block sequentially. Consider using `nn.Sequential`
3. create a layer norm layer
4. create a linear layer for predicting the next token

`forward(self, idx, targets=None)`. (5 points)

`forward` takes a batch of context ids as input of size (B, T) and returns the logits and the loss, if targets is not None. If targets is None, return the logits and None.
1. get the token by using the token embedding table created in the constructor
2. create the position embeddings
3. sum the token and position embeddings to get the model input
4. pass the model through the blocks, the layernorm layer, and the final linear layer
5. compute the loss

`generate(start_char, max_new_tokens, top_p, top_k, temperature) -> str` (5 points)
1. implement top p, top_k, and temperature for sampling



![](https://i.ibb.co/n8sbQ0V/Screenshot-2024-01-23-at-8-59-08-PM.png)

In [ ]:
#GPT
class GPT(nn.Module):
    def __init__(self, n_embd, n_head):
        super().__init__()
        #Create the token embedding table and the position embedding table
        self.token_embedding_table = nn.Embedding(len(chars), n_embd)
        self.position_embedding_table = nn.Embedding(block_size, n_embd)

        #Create variable self.blocks that is a series of 4 Blocks. The data will pass through each block sequentially
        self.blocks = nn.Sequential(
            Block(n_embd, n_head),
            Block(n_embd, n_head),
            Block(n_embd, n_head),
            Block(n_embd, n_head)
        )

        #Create a layer norm layer
        self.ln_f = nn.LayerNorm(n_embd)
        self.lm_head = nn.Linear(n_embd, len(chars))

    def forward(self, idx, targets=None):
      #Get the token embeddings using the token embedding table created in the constructor
        B, T = idx.shape
        tok_emb = self.token_embedding_table(idx)

        #Create the position embeddings
        pos_emb = self.position_embedding_table(torch.arange(T, device=idx.device))

        #Sum the token and position embeddings to get the model input
        x = tok_emb + pos_emb

        #Pass the model input through the blocks, layernorm, and final linear layer
        x = self.blocks(x)
        x = self.ln_f(x)
        logits = self.lm_head(x)

        #Compute the loss if targets are provided
        loss = None
        if targets is not None:
            logits = logits.view(-1, logits.size(-1))
            targets = targets.view(-1)
            loss = F.cross_entropy(logits, targets)

        return logits, loss

    def generate(self, start_char, max_new_tokens, top_p=1.0, top_k=None, temperature=1.0):
        #Generate method with top_p, top_k, and temperature-based sampling
        if start_char not in chars:
            raise ValueError(f"'{start_char}' not in the vocabulary")

        start_id = chars.index(start_char)
        idx = torch.tensor([[start_id]], device=device)

        for _ in range(max_new_tokens):
            idx_cond = idx[:, -block_size:]
            logits, _ = self(idx_cond)
            logits = logits[:, -1, :] / temperature

            if top_k is not None:
                v, ix = torch.topk(logits, top_k)
                logits[logits < v[:, [-1]]] = -float('Inf')

            probs = F.softmax(logits, dim=-1)

            if top_p < 1.0:
                sorted_probs, sorted_idx = torch.sort(probs, descending=True)
                cumulative_probs = torch.cumsum(sorted_probs, dim=-1)
                sorted_mask = cumulative_probs > top_p
                sorted_probs[sorted_mask] = 0
                sorted_probs = sorted_probs / sorted_probs.sum(dim=-1, keepdim=True)
                probs = torch.zeros_like(probs).scatter(1, sorted_idx, sorted_probs)

            if torch.isnan(probs).any() or torch.sum(probs) <= 0:
                raise ValueError("Sampling distribution became invalid. Try adjusting top_p/top_k.")

            next_id = torch.multinomial(probs, num_samples=1)
            idx = torch.cat((idx, next_id), dim=1)

        return decode(idx[0].tolist())

### Training loop (15 points)

implement training loop

In [ ]:
#Training loop
model = GPT(n_embd=n_embd, n_head=n_head).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

max_iters = 5000
eval_interval = 500

for step in range(max_iters):
    xb, yb = get_batch()
    logits, loss = model(xb, yb)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    if step % eval_interval == 0:
        print(f"Step {step}: Loss = {loss.item():.4f}")

Step 0: Loss = 4.2990
Step 500: Loss = 2.3521
Step 1000: Loss = 2.2077
Step 1500: Loss = 2.1404
Step 2000: Loss = 2.0560
Step 2500: Loss = 2.0196
Step 3000: Loss = 2.0186
Step 3500: Loss = 1.9936
Step 4000: Loss = 2.0329
Step 4500: Loss = 1.9631


### Generate text


print some text that your model generates

In [ ]:
print("\nGenerated text:\n")
print(model.generate(start_char='T',
                     max_new_tokens=500,
                     top_p=1.9,
                     top_k=2,
                     temperature=1))


Generated text:

TIUS:
A man and to to meed the will him the soned the so dook,
And what we shall a would to the here and, and they when well to him with strower she will he have to me he here he had the seet the with than the some have the marry heaved and her with the seepere and he have my like he with and will to mare, the here a many sone, and the wither to me,
To to me have will here, to the weend, and this wich are the with, and we when the seall the stand with whe wice to the head thing hand, the will the
